# Gap de tasas: Finagro vs no-Finagro por producto

Notebook auto-contenido (corre en Colab) que estima la **brecha de tasa de interés** entre crédito Finagro (lower-bound: `is_fag & is_redescuento`) y crédito no-Finagro, por `producto_de_credito_red`, y su **evolución en el tiempo** vía `fecha_corte`.

Tasa = **promedio ponderado por `montos_desembolsados`** (más fiel al costo real del crédito que el promedio simple).

Fuente: dataset SFC consolidado (`datos.csv`, ~1.55 GB), descargado del mismo Drive ID que [Consolidación_Finagro_y_SFC.ipynb](Consolidación_Finagro_y_SFC.ipynb).

## 1. Setup

In [40]:
!pip install -q gdown altair

In [41]:
import pandas as pd
import numpy as np
import altair as alt

alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [42]:
!gdown --id '1pqULCPNUfiZ1H9bvtahu2gJOBbHPnVMr' --output datos.csv

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1pqULCPNUfiZ1H9bvtahu2gJOBbHPnVMr
From (redirected): https://drive.google.com/uc?id=1pqULCPNUfiZ1H9bvtahu2gJOBbHPnVMr&confirm=t&uuid=90b6db79-1e8f-4b76-8502-cce38643266a
To: /content/datos.csv
100% 1.55G/1.55G [00:32<00:00, 48.4MB/s]


In [43]:
df = pd.read_csv('datos.csv')
print(df.shape)
df.head()

(3339047, 34)


,tipo_entidad,nombre_tipo_entidad,codigo_entidad,nombre_entidad,fecha_corte,tipo_de_persona,sexo,tama_o_de_empresa,tipo_de_cr_dito,tipo_de_garant_a,...,tasa_ponderada,mes,year,year_month,costo_credito,costo_credito_ponderado,cod_departamento,nombre_departamento,nombre_municipio,tipo_municipio
0,1,BC-ESTABLECIMIENTO BANCARIO,7,Bancolombia,2025-02-28,Natural,Femenino,Microempresa,Crédito productivo,Sin garantia,...,251.52,2,2025,2025-02,20.96,20.96,76.0,VALLE DEL CAUCA,TULUÁ,Ciudades y aglomeraciones
1,1,BC-ESTABLECIMIENTO BANCARIO,7,Bancolombia,2025-02-28,Natural,Femenino,Microempresa,Crédito productivo,Sin garantia,...,268.80,2,2025,2025-02,22.40,22.40,5.0,ANTIOQUIA,MEDELLÍN,Ciudades y aglomeraciones
2,1,BC-ESTABLECIMIENTO BANCARIO,7,Bancolombia,2025-02-28,Natural,Femenino,Microempresa,Crédito productivo,Sin garantia,...,354.48,2,2025,2025-02,29.54,29.54,5.0,ANTIOQUIA,BELLO,Ciudades y aglomeraciones
3,1,BC-ESTABLECIMIENTO BANCARIO,7,Bancolombia,2025-02-28,Natural,Femenino,Microempresa,Crédito productivo,Sin garantia,...,251.52,2,2025,2025-02,20.96,20.96,63.0,QUINDÍO,ARMENIA,Ciudades y aglomeraciones
4,1,BC-ESTABLECIMIENTO BANCARIO,7,Bancolombia,2025-02-28,Natural,Femenino,Microempresa,Crédito productivo,Sin garantia,...,354.48,2,2025,2025-02,29.54,29.54,76.0,VALLE DEL CAUCA,SANTIAGO DE CALI,Ciudades y aglomeraciones


## 2. Prep: banderas y mapeo de producto

Reproduce verbatim el bloque de prep del notebook fuente: normaliza texto y fecha, deriva `is_fag`, `is_redescuento`, `is_finagro_lb` y mapea `producto_de_cr_dito` → `producto_de_credito_red` (5 categorías).

In [44]:
df["fecha_corte"] = pd.to_datetime(df["fecha_corte"], errors="coerce")

for col in ["producto_de_cr_dito", "tipo_de_garant_a"]:
    df[col] = df[col].astype(str).str.strip().str.lower()

df["tasa_efectiva_promedio"] = pd.to_numeric(df["tasa_efectiva_promedio"], errors="coerce")
df["montos_desembolsados"]   = pd.to_numeric(df["montos_desembolsados"], errors="coerce").fillna(0)
df["numero_de_creditos"]     = pd.to_numeric(df["numero_de_creditos"], errors="coerce").fillna(0)

df["is_redescuento"] = df["producto_de_cr_dito"].str.contains("con recursos de redescuento", na=False)
df["is_fag"]         = df["tipo_de_garant_a"].str.contains("fag", na=False)
df["is_finagro_lb"]  = df["is_fag"] & df["is_redescuento"]

df[["is_fag", "is_redescuento", "is_finagro_lb"]].sum()

,0
is_fag,484464
is_redescuento,514133
is_finagro_lb,473144


In [45]:
df['producto_de_credito_red'] = df['producto_de_cr_dito'].replace({
    'crédito productivo urbano (sin recursos de redescuento)': 'Crédito productivo urbano',
    'crédito productivo urbano  (con recursos de redescuento)': 'Crédito productivo urbano',
    'crédito productivo rural (sin recursos de redescuento)': 'Crédito productivo rural',
    'crédito productivo rural  (con recursos de redescuento)': 'Crédito productivo rural',
    'crédito popular productivo urbano (sin recursos de redescuento)': 'Crédito popular productivo urbano',
    'crédito popular productivo urbano  (con recursos de redescuento)': 'Crédito popular productivo urbano',
    'crédito popular productivo rural (sin recursos de redescuento)': 'Crédito popular productivo rural',
    'crédito popular productivo rural (con recursos de redescuento)': 'Crédito popular productivo rural',
    'crédito productivo de mayor monto (sin recursos de redescuento)': 'Crédito productivo de mayor monto',
    'crédito productivo de mayor monto (con recursos de redescuento)': 'Crédito productivo de mayor monto',
})

PRODUCTOS = [
    'Crédito popular productivo rural', 'Crédito popular productivo urbano',
    'Crédito productivo de mayor monto', 'Crédito productivo rural',
    'Crédito productivo urbano',
]
df = df[df['producto_de_credito_red'].isin(PRODUCTOS)].copy()
df['producto_de_credito_red'].value_counts()

,count
producto_de_credito_red,
Crédito popular productivo urbano,1726815
Crédito productivo urbano,658183
Crédito popular productivo rural,461992
Crédito productivo rural,323871
Crédito productivo de mayor monto,168186


## 3. Estadística descriptiva por producto e is_finagro_lb

Antes de mirar el gap, ¿qué tan grande es cada combinación? Total de monto desembolsado, número de créditos y participación (share) Finagro vs no-Finagro dentro de cada producto.

In [46]:
desc = (df.groupby(["producto_de_credito_red", "is_finagro_lb"], as_index=False)
          .agg(monto_total=("montos_desembolsados", "sum"),
               creditos_total=("numero_de_creditos", "sum"),
               n_obs=("fecha_corte", "size")))
desc["share_monto"]    = desc.groupby("producto_de_credito_red")["monto_total"].transform(lambda s: s / s.sum())
desc["share_creditos"] = desc.groupby("producto_de_credito_red")["creditos_total"].transform(lambda s: s / s.sum())
desc.sort_values(["producto_de_credito_red", "is_finagro_lb"])

,producto_de_credito_red,is_finagro_lb,monto_total,creditos_total,n_obs,share_monto,share_creditos
0,Crédito popular productivo rural,False,1.420911e+12,404940,397053,0.807635,0.857030
1,Crédito popular productivo rural,True,3.384379e+11,67552,64939,0.192365,0.142970
2,Crédito popular productivo urbano,False,5.819483e+12,1726411,1667823,0.950010,0.965609
3,Crédito popular productivo urbano,True,3.062267e+11,61487,58992,0.049990,0.034391
4,Crédito productivo de mayor monto,False,7.354753e+12,135099,132878,0.817354,0.786552
5,Crédito productivo de mayor monto,True,1.643489e+12,36662,35308,0.182646,0.213448
6,Crédito productivo rural,False,2.027360e+12,145050,142393,0.364308,0.412964
7,Crédito productivo rural,True,3.537605e+12,206191,181478,0.635692,0.587036
8,Crédito productivo urbano,False,8.047418e+12,543610,525756,0.762817,0.784493
9,Crédito productivo urbano,True,2.502185e+12,149334,132427,0.237183,0.215507


In [47]:
finagro_color = alt.Scale(
    domain=[True, False],
    range=["#1f77b4", "#d62728"],
)

monto_chart = (
    alt.Chart(desc)
    .mark_bar()
    .encode(
        y=alt.Y("producto_de_credito_red:N", title=None, sort="-x"),
        x=alt.X("monto_total:Q",
                title="Monto desembolsado acumulado (COP)",
                axis=alt.Axis(format="~s")),
        color=alt.Color("is_finagro_lb:N", title="¿Finagro (lb)?", scale=finagro_color),
        yOffset="is_finagro_lb:N",
        tooltip=[
            "producto_de_credito_red:N", "is_finagro_lb:N",
            alt.Tooltip("monto_total:Q", format=",.0f"),
            alt.Tooltip("share_monto:Q", format=".1%"),
        ],
    )
    .properties(width=600, height=200, title="Monto desembolsado por producto y origen")
)
monto_chart

alt.Chart(...)

In [48]:
creditos_chart = (
    alt.Chart(desc)
    .mark_bar()
    .encode(
        y=alt.Y("producto_de_credito_red:N", title=None, sort="-x"),
        x=alt.X("creditos_total:Q", title="Número de créditos (acumulado)", axis=alt.Axis(format="~s")),
        color=alt.Color("is_finagro_lb:N", title="¿Finagro (lb)?", scale=finagro_color),
        yOffset="is_finagro_lb:N",
        tooltip=[
            "producto_de_credito_red:N", "is_finagro_lb:N",
            alt.Tooltip("creditos_total:Q", format=",.0f"),
            alt.Tooltip("share_creditos:Q", format=".1%"),
        ],
    )
    .properties(width=600, height=200, title="Número de créditos por producto y origen")
)
creditos_chart

alt.Chart(...)

### Evolución temporal de montos y número de créditos

¿La participación Finagro crece, decae o es estacional? Series de tiempo por `fecha_corte`, con eje Y independiente por panel para que productos chicos no queden aplastados.

In [49]:
ts_desc = (df.groupby(["fecha_corte", "producto_de_credito_red", "is_finagro_lb"], as_index=False)
             .agg(monto=("montos_desembolsados", "sum"),
                  creditos=("numero_de_creditos", "sum")))

base = alt.Chart(ts_desc).encode(
    x=alt.X("fecha_corte:T", title="Fecha de corte"),
    color=alt.Color("is_finagro_lb:N", title="¿Finagro (lb)?", scale=finagro_color),
    tooltip=["fecha_corte:T", "producto_de_credito_red:N", "is_finagro_lb:N",
             alt.Tooltip("monto:Q", format=",.0f"),
             alt.Tooltip("creditos:Q", format=",.0f")],
)

monto_ts = (base.mark_area(opacity=0.6)
            .encode(y=alt.Y("monto:Q", stack=None, title="Monto desembolsado (COP)",
                            axis=alt.Axis(format="~s")))
            .properties(width=380, height=180, title="Monto desembolsado")
            .facet(facet=alt.Facet("producto_de_credito_red:N", title=None), columns=2)
            .resolve_scale(y="independent"))

creditos_ts = (base.mark_area(opacity=0.6)
               .encode(y=alt.Y("creditos:Q", stack=None, title="N° de créditos",
                               axis=alt.Axis(format="~s")))
               .properties(width=380, height=180, title="Número de créditos")
               .facet(facet=alt.Facet("producto_de_credito_red:N", title=None), columns=2)
               .resolve_scale(y="independent"))

monto_ts & creditos_ts

alt.VConcatChart(...)

## 4. Tasa ponderada por monto

Para cada `(fecha_corte, producto_de_credito_red, is_finagro_lb)`:

$$\text{tasa\_ponderada} = \frac{\sum_i \text{tasa\_efectiva\_promedio}_i \cdot \text{montos\_desembolsados}_i}{\sum_i \text{montos\_desembolsados}_i}$$

Filtramos filas con tasa nula o monto cero (no aportan al ponderado).

In [50]:
mask = df["tasa_efectiva_promedio"].notna() & (df["montos_desembolsados"] > 0)
sub = df.loc[mask, [
    "fecha_corte", "producto_de_credito_red", "is_finagro_lb",
    "tasa_efectiva_promedio", "montos_desembolsados",
]].copy()
sub["num"] = sub["tasa_efectiva_promedio"] * sub["montos_desembolsados"]

g = (sub.groupby(["fecha_corte", "producto_de_credito_red", "is_finagro_lb"], as_index=False)
        .agg(num=("num", "sum"), den=("montos_desembolsados", "sum")))
g["tasa_ponderada"] = g["num"] / g["den"]
g.head()

,fecha_corte,producto_de_credito_red,is_finagro_lb,num,den,tasa_ponderada
0,2023-09-29,Crédito popular productivo rural,False,4.397347e+11,9.156941e+09,48.022013
1,2023-09-29,Crédito popular productivo rural,True,4.571077e+10,4.001975e+09,11.422052
2,2023-09-29,Crédito popular productivo urbano,False,2.217783e+12,4.544496e+10,48.801517
3,2023-09-29,Crédito popular productivo urbano,True,3.948565e+10,3.509224e+09,11.251960
4,2023-09-29,Crédito productivo de mayor monto,False,2.217579e+12,7.258531e+10,30.551349


## 5. Pivot a tasa_finagro / tasa_no_finagro / gap

In [51]:
wide = (
    g.pivot_table(
        index=["fecha_corte", "producto_de_credito_red"],
        columns="is_finagro_lb",
        values="tasa_ponderada",
    )
    .rename(columns={True: "tasa_finagro", False: "tasa_no_finagro"})
    .reset_index()
)
wide.columns.name = None
wide["gap"] = wide["tasa_no_finagro"] - wide["tasa_finagro"]
wide = wide.dropna(subset=["tasa_finagro", "tasa_no_finagro"])
wide.head()

,fecha_corte,producto_de_credito_red,tasa_no_finagro,tasa_finagro,gap
0,2023-09-29,Crédito popular productivo rural,48.022013,11.422052,36.599960
1,2023-09-29,Crédito popular productivo urbano,48.801517,11.251960,37.549557
2,2023-09-29,Crédito productivo de mayor monto,30.551349,10.814541,19.736807
3,2023-09-29,Crédito productivo rural,38.389015,9.582813,28.806202
4,2023-09-29,Crédito productivo urbano,40.323802,9.668793,30.655009


## 6. Evolución temporal del gap por producto

Tres líneas por panel: **tasa Finagro** (azul), **tasa no-Finagro** (rojo), **gap** (verde). Eje Y independiente por panel — el rango de tasas y del gap difiere mucho entre productos.

In [52]:
long = wide.melt(
    id_vars=["fecha_corte", "producto_de_credito_red"],
    value_vars=["tasa_finagro", "tasa_no_finagro", "gap"],
    var_name="serie", value_name="tasa",
)

color_scale = alt.Scale(
    domain=["tasa_finagro", "tasa_no_finagro", "gap"],
    range=["#1f77b4", "#d62728", "#2ca02c"],
)

chart = (
    alt.Chart(long)
    .mark_line(point=True)
    .encode(
        x=alt.X("fecha_corte:T", title="Fecha de corte"),
        y=alt.Y("tasa:Q", title="Tasa ponderada (%)", axis=alt.Axis(format=".1f")),
        color=alt.Color("serie:N", scale=color_scale, title=None),
        tooltip=[
            "fecha_corte:T", "producto_de_credito_red:N", "serie:N",
            alt.Tooltip("tasa:Q", format=".2f", title="Tasa (%)"),
        ],
    )
    .properties(width=380, height=220)
    .facet(
        facet=alt.Facet("producto_de_credito_red:N", title=None),
        columns=2,
    )
    .resolve_scale(y="independent")
)
chart

alt.FacetChart(...)

## 7. Tabla resumen del gap por producto

In [53]:
(wide.groupby("producto_de_credito_red")
     .agg(gap_promedio=("gap", "mean"),
          gap_min=("gap", "min"),
          gap_max=("gap", "max"),
          n_semanas=("fecha_corte", "nunique"))
     .sort_values("gap_promedio", ascending=False))

,gap_promedio,gap_min,gap_max,n_semanas
producto_de_credito_red,,,,
Crédito popular productivo urbano,45.445714,33.185188,54.252948,118
Crédito popular productivo rural,43.550157,32.237597,51.195535,118
Crédito productivo urbano,29.740453,25.483584,37.898863,118
Crédito productivo de mayor monto,16.190550,11.931973,23.201652,118
Crédito productivo rural,14.953831,6.434997,30.236348,118


## 8. Descomposición del cambio en `tasa_no_finagro` para Crédito productivo rural

Único producto donde Finagro tiene mayoría de mercado (~64% del monto) y único donde el gap se está cerrando. Pregunta: ¿la caída de la tasa no-Finagro rural viene de (a) **cada entidad bajando su tasa** (efecto **within**), de (b) **reasignación de cuota** hacia entidades más baratas (efecto **between**), o de (c) **entradas/salidas** de entidades?

Descomposición shift-share entre la ventana **inicial** (primeras 12 semanas) y la **final** (últimas 12 semanas):

$$\Delta \bar{r} = \underbrace{\sum_e w_e^0 \cdot \Delta r_e}_{\text{within}} + \underbrace{\sum_e r_e^0 \cdot \Delta w_e}_{\text{between}} + \underbrace{\sum_e \Delta w_e \cdot \Delta r_e}_{\text{cross}} + \text{entrants} - \text{exits}$$

donde $w_e$ es la cuota de monto desembolsado de la entidad $e$ y $r_e$ su tasa ponderada por monto.

In [54]:
PROD = "Crédito productivo rural"

ent_mask = (
    (df["producto_de_credito_red"] == PROD)
    & (~df["is_finagro_lb"])
    & df["tasa_efectiva_promedio"].notna()
    & (df["montos_desembolsados"] > 0)
    & df["nombre_entidad"].notna()
)

ent = df.loc[ent_mask, [
    "fecha_corte", "nombre_entidad", "tasa_efectiva_promedio", "montos_desembolsados",
]].copy()
ent["nombre_entidad"] = ent["nombre_entidad"].astype(str).str.strip()
ent["num"] = ent["tasa_efectiva_promedio"] * ent["montos_desembolsados"]

print(f"Filas: {len(ent):,}")
print(f"Entidades únicas: {ent['nombre_entidad'].nunique()}")
print(f"Rango fechas: {ent['fecha_corte'].min().date()} -> {ent['fecha_corte'].max().date()}")

Filas: 142,393
Entidades únicas: 17
Rango fechas: 2023-09-29 -> 2025-12-26


In [55]:
W = 12  # ventana en semanas
fechas = sorted(ent["fecha_corte"].unique())
ventana_0 = fechas[:W]
ventana_T = fechas[-W:]
print(f"Ventana INICIAL: {pd.Timestamp(ventana_0[0]).date()} -> {pd.Timestamp(ventana_0[-1]).date()}  ({W} cortes)")
print(f"Ventana FINAL:   {pd.Timestamp(ventana_T[0]).date()} -> {pd.Timestamp(ventana_T[-1]).date()}  ({W} cortes)")

Ventana INICIAL: 2023-09-29 -> 2023-12-15  (12 cortes)
Ventana FINAL:   2025-10-10 -> 2025-12-26  (12 cortes)


In [56]:
def agg_ventana(df_ent, fechas_ventana):
    g = (df_ent[df_ent["fecha_corte"].isin(fechas_ventana)]
            .groupby("nombre_entidad", as_index=False)
            .agg(num=("num", "sum"), monto=("montos_desembolsados", "sum")))
    g["tasa"]  = g["num"] / g["monto"]
    g["share"] = g["monto"] / g["monto"].sum()
    return g[["nombre_entidad", "monto", "tasa", "share"]]

agg_0 = agg_ventana(ent, ventana_0).rename(columns={"monto": "monto_0", "tasa": "tasa_0", "share": "share_0"})
agg_T = agg_ventana(ent, ventana_T).rename(columns={"monto": "monto_T", "tasa": "tasa_T", "share": "share_T"})

merged = agg_0.merge(agg_T, on="nombre_entidad", how="outer")
merged["estado"] = "ambas"
merged.loc[merged["share_0"].isna(), "estado"] = "entrant"
merged.loc[merged["share_T"].isna(), "estado"] = "exit"

tasa_mercado_0 = (merged["share_0"] * merged["tasa_0"]).sum()
tasa_mercado_T = (merged["share_T"] * merged["tasa_T"]).sum()
delta_total    = tasa_mercado_T - tasa_mercado_0

print(f"Tasa mercado inicial: {tasa_mercado_0:.2f} %")
print(f"Tasa mercado final:   {tasa_mercado_T:.2f} %")
print(f"Δ total:              {delta_total:+.2f} pp")
print()
print(merged["estado"].value_counts())

Tasa mercado inicial: 39.32 %
Tasa mercado final:   25.21 %
Δ total:              -14.10 pp

estado
ambas      12
entrant     3
exit        1
Name: count, dtype: int64


In [57]:
both     = merged[merged["estado"] == "ambas"].copy()
entrants = merged[merged["estado"] == "entrant"].copy()
exits    = merged[merged["estado"] == "exit"].copy()

both["within"]  = both["share_0"] * (both["tasa_T"] - both["tasa_0"])
both["between"] = both["tasa_0"]  * (both["share_T"] - both["share_0"])
both["cross"]   = (both["share_T"] - both["share_0"]) * (both["tasa_T"] - both["tasa_0"])

entry_contrib =  (entrants["share_T"] * entrants["tasa_T"]).sum()
exit_contrib  = -(exits["share_0"]    * exits["tasa_0"]).sum()

resumen = pd.DataFrame({
    "componente": [
        "Within (entidades cambian su tasa)",
        "Between (reasignación de cuota)",
        "Cross (interacción)",
        "Entrants (entidades nuevas)",
        "Exits (entidades que salen)",
        "TOTAL (suma)",
        "TOTAL (observado)",
    ],
    "contribucion_pp": [
        both["within"].sum(),
        both["between"].sum(),
        both["cross"].sum(),
        entry_contrib,
        exit_contrib,
        both["within"].sum() + both["between"].sum() + both["cross"].sum() + entry_contrib + exit_contrib,
        delta_total,
    ],
})
resumen["contribucion_pp"]   = resumen["contribucion_pp"].round(3)
resumen["share_del_total_%"] = (resumen["contribucion_pp"] / delta_total * 100).round(1)
resumen

,componente,contribucion_pp,share_del_total_%
0,Within (entidades cambian su tasa),-13.900,98.6
1,Between (reasignación de cuota),-1.646,11.7
2,Cross (interacción),0.230,-1.6
3,Entrants (entidades nuevas),2.747,-19.5
4,Exits (entidades que salen),-1.535,10.9
5,TOTAL (suma),-14.104,100.0
6,TOTAL (observado),-14.104,100.0


In [58]:
wf_data = pd.DataFrame({
    "componente": ["Within", "Between", "Cross", "Entrants", "Exits"],
    "contribucion": [
        both["within"].sum(), both["between"].sum(), both["cross"].sum(),
        entry_contrib, exit_contrib,
    ],
})

wf_chart = (
    alt.Chart(wf_data)
    .mark_bar()
    .encode(
        x=alt.X("componente:N", sort=None, title=None),
        y=alt.Y("contribucion:Q", title="Contribución a Δ tasa (pp)"),
        color=alt.condition(
            "datum.contribucion < 0",
            alt.value("#2ca02c"),
            alt.value("#d62728"),
        ),
        tooltip=[alt.Tooltip("componente:N"), alt.Tooltip("contribucion:Q", format=".2f")],
    )
    .properties(width=500, height=300,
                title=f"Descomposición Δ tasa_no_finagro rural: {delta_total:+.2f} pp")
)
wf_chart

alt.Chart(...)

### Top entidades por contribución al efecto **within**

¿Qué entidades son las que más están moviendo su propia tasa (ponderado por su cuota inicial)?

In [59]:
top_within = (
    both.assign(abs_within=both["within"].abs())
        .nlargest(15, "abs_within")
        [["nombre_entidad", "share_0", "share_T", "tasa_0", "tasa_T",
          "within", "between", "cross"]]
        .sort_values("within")
        .round({"share_0": 4, "share_T": 4, "tasa_0": 2, "tasa_T": 2,
                "within": 3, "between": 3, "cross": 3})
)
top_within

,nombre_entidad,share_0,share_T,tasa_0,tasa_T,within,between,cross
4,Banco Mundo Mujer S.A.,0.2097,0.2015,45.69,26.99,-3.921,-0.373,0.153
1,Bancamía S.A.,0.1475,0.0895,44.12,26.91,-2.538,-2.559,0.998
0,Banagrario,0.3103,0.2119,27.43,20.92,-2.018,-2.699,0.640
6,Banco W S.A.,0.0800,0.1006,46.04,26.72,-1.545,0.951,-0.399
13,Crezcamos,0.0864,0.0465,43.76,26.56,-1.486,-1.747,0.686
15,Mibanco S.A.,0.0641,0.0384,46.37,26.63,-1.264,-1.190,0.506
5,Banco Santander,0.0316,0.1234,46.74,26.75,-0.632,4.291,-1.836
2,Banco Caja Social S.A.,0.0195,0.0338,39.69,20.91,-0.367,0.565,-0.267
11,Cooperativa Financiera de Antioquia,0.0069,0.0090,33.81,23.55,-0.071,0.072,-0.022
8,Bancolombia,0.0065,0.0408,30.57,23.86,-0.044,1.048,-0.230


### Evolución temporal por entidad (top 8 por monto acumulado)

Para entender si el patrón es generalizado o concentrado en pocas entidades.

In [60]:
top_ents = (
    ent.groupby("nombre_entidad")["montos_desembolsados"].sum()
       .nlargest(8).index.tolist()
)

ts_ent = (
    ent[ent["nombre_entidad"].isin(top_ents)]
        .groupby(["fecha_corte", "nombre_entidad"], as_index=False)
        .agg(num=("num", "sum"), monto=("montos_desembolsados", "sum"))
)
ts_ent["tasa"] = ts_ent["num"] / ts_ent["monto"]

tasa_por_entidad = (
    alt.Chart(ts_ent)
    .mark_line()
    .encode(
        x=alt.X("fecha_corte:T", title="Fecha de corte"),
        y=alt.Y("tasa:Q", title="Tasa ponderada (%)", axis=alt.Axis(format=".1f")),
        color=alt.Color("nombre_entidad:N", title="Entidad"),
        tooltip=["fecha_corte:T", "nombre_entidad:N",
                 alt.Tooltip("tasa:Q", format=".2f"),
                 alt.Tooltip("monto:Q", format=",.0f")],
    )
    .properties(width=700, height=320,
                title="Tasa no-Finagro rural por entidad (top 8 por monto)")
)
tasa_por_entidad

alt.Chart(...)

In [61]:
weekly_total = (ent.groupby("fecha_corte")["montos_desembolsados"].sum()
                   .rename("monto_total_semana").reset_index())

share_ts = (
    ent[ent["nombre_entidad"].isin(top_ents)]
        .groupby(["fecha_corte", "nombre_entidad"], as_index=False)["montos_desembolsados"].sum()
        .merge(weekly_total, on="fecha_corte")
)
share_ts["share"] = share_ts["montos_desembolsados"] / share_ts["monto_total_semana"]

share_chart = (
    alt.Chart(share_ts)
    .mark_area(opacity=0.75)
    .encode(
        x=alt.X("fecha_corte:T", title="Fecha de corte"),
        y=alt.Y("share:Q", stack="normalize",
                title="Cuota de monto (top 8 entidades)",
                axis=alt.Axis(format=".0%")),
        color=alt.Color("nombre_entidad:N", title="Entidad"),
        tooltip=["fecha_corte:T", "nombre_entidad:N",
                 alt.Tooltip("share:Q", format=".1%"),
                 alt.Tooltip("montos_desembolsados:Q", format=",.0f")],
    )
    .properties(width=700, height=320,
                title="Cuota de mercado no-Finagro rural por entidad (top 8)")
)
share_chart

alt.Chart(...)

## 9. Regresión: ¿es la tendencia rural distinta del resto, controlando por macro y por la tasa Finagro?

Estimamos por producto:

$$\text{tasa\_total}_{t,p} = \alpha_p + \beta_{\text{time},p}\,t + \beta_{DTF,p}\,\text{DTF}_t + \beta_{IBR,p}\,\text{IBR}_t + \beta_{TES,p}\,\text{TES}_t + \beta_{F,p}\,\text{tasa\_finagro}_{t,p} + \varepsilon_{t,p}$$

con SE Newey-West (HAC, lags=4). Comparamos $\beta_{\text{time}}$, $\beta_F$ y betas macro entre los 5 productos.

**Variable dependiente**: `tasa_total` = promedio ponderado por monto desembolsado sobre **todas** las observaciones del producto en cada semana (sin distinguir Finagro/no-Finagro).

⚠ **Caveat sobre `β_finagro`**: como `tasa_total ≈ w·tasa_finagro + (1−w)·tasa_no_finagro`, el coeficiente mezcla composición (peso `w`) con pass-through real. Lectura **comparativa entre productos** (orden y magnitud relativa), no causal puntual.


In [62]:
!pip install -q pymannkendall

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pymannkendall as mk

In [63]:
!gdown --id '1GGsdRufXrUE651qFNEeTEepRJnDDK3B3' --output tasas_banrep_semanal.csv

macro = pd.read_csv("tasas_banrep_semanal.csv")
macro["fecha"] = pd.to_datetime(macro["fecha"], errors="coerce")
macro = macro.rename(columns={"fecha": "fecha_corte"})
print("Cobertura macro:", macro["fecha_corte"].min().date(), "->", macro["fecha_corte"].max().date())
print("Nulos en rango wide:")
rango = macro[(macro["fecha_corte"] >= wide["fecha_corte"].min()) & (macro["fecha_corte"] <= wide["fecha_corte"].max())]
print(rango[["dtf_90d", "ibr_overnight", "tes_1y"]].isna().sum())

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1GGsdRufXrUE651qFNEeTEepRJnDDK3B3
To: /content/tasas_banrep_semanal.csv
100% 50.3k/50.3k [00:00<00:00, 32.6MB/s]
Cobertura macro: 1984-01-20 -> 2026-05-08
Nulos en rango wide:
dtf_90d          0
ibr_overnight    0
tes_1y           0
dtype: int64


In [64]:
tot_mask = (df["tasa_efectiva_promedio"].notna()
            & (df["montos_desembolsados"] > 0))
tot = df.loc[tot_mask, [
    "fecha_corte", "producto_de_credito_red",
    "tasa_efectiva_promedio", "montos_desembolsados",
]].copy()
tot["num"] = tot["tasa_efectiva_promedio"] * tot["montos_desembolsados"]

panel = (tot.groupby(["fecha_corte", "producto_de_credito_red"], as_index=False)
            .agg(num=("num", "sum"), monto=("montos_desembolsados", "sum")))
panel["tasa_total"] = panel["num"] / panel["monto"]

# Merge macro (semanal viernes a viernes)
panel = panel.merge(macro[["fecha_corte", "dtf_90d", "ibr_overnight", "tes_1y"]],
                    on="fecha_corte", how="left")

# Merge tasa_finagro per (fecha, producto) desde wide
panel = panel.merge(wide[["fecha_corte", "producto_de_credito_red", "tasa_finagro"]],
                    on=["fecha_corte", "producto_de_credito_red"], how="left")

panel["t"] = ((panel["fecha_corte"] - panel["fecha_corte"].min()).dt.days // 7).astype(int)
panel = panel.dropna(subset=["dtf_90d", "ibr_overnight", "tes_1y",
                             "tasa_total", "tasa_finagro"])
print("panel shape:", panel.shape)
print(panel.groupby("producto_de_credito_red").size().rename("n_semanas"))
panel.head()

panel shape: (590, 10)
producto_de_credito_red
Crédito popular productivo rural     118
Crédito popular productivo urbano    118
Crédito productivo de mayor monto    118
Crédito productivo rural             118
Crédito productivo urbano            118
Name: n_semanas, dtype: int64


,fecha_corte,producto_de_credito_red,num,monto,tasa_total,dtf_90d,ibr_overnight,tes_1y,tasa_finagro,t
0,2023-09-29,Crédito popular productivo rural,4.854455e+11,1.315892e+10,36.890994,13.01,12.287,10.6,11.422052,0
1,2023-09-29,Crédito popular productivo urbano,2.257269e+12,4.895419e+10,46.109820,13.01,12.287,10.6,11.251960,0
2,2023-09-29,Crédito productivo de mayor monto,2.432804e+12,9.248671e+10,26.304359,13.01,12.287,10.6,10.814541,0
3,2023-09-29,Crédito productivo rural,8.360248e+11,5.156090e+10,16.214316,13.01,12.287,10.6,9.582813,0
4,2023-09-29,Crédito productivo urbano,3.067182e+12,9.651817e+10,31.778284,13.01,12.287,10.6,9.668793,0


In [65]:
def fit_one(prod_df,
            formula_cols=("t", "dtf_90d", "ibr_overnight", "tes_1y", "tasa_finagro"),
            maxlags=4):
    X = sm.add_constant(prod_df[list(formula_cols)].values)
    y = prod_df["tasa_total"].values
    res = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": maxlags})
    return res

def coef_table(res, names):
    ci = res.conf_int()
    return pd.DataFrame({
        "var":   ["const"] + list(names),
        "beta":  res.params,
        "se":    res.bse,
        "t":     res.tvalues,
        "p":     res.pvalues,
        "ci_lo": ci[:, 0],
        "ci_hi": ci[:, 1],
    })

In [66]:
NAMES = ["t", "dtf_90d", "ibr_overnight", "tes_1y", "tasa_finagro"]

rows = []
for prod in PRODUCTOS:
    sub = panel[panel["producto_de_credito_red"] == prod].sort_values("fecha_corte")
    if len(sub) < len(NAMES) + 5:
        print(f"Saltando {prod}: solo {len(sub)} obs.")
        continue
    res = fit_one(sub)
    tbl = coef_table(res, NAMES).assign(
        producto=prod, n=int(res.nobs),
        r2=res.rsquared, r2_adj=res.rsquared_adj,
    )
    rows.append(tbl)

betas = pd.concat(rows, ignore_index=True)
betas[["producto", "var", "beta", "se", "p", "ci_lo", "ci_hi", "n", "r2_adj"]].round(4)

,producto,var,beta,se,p,ci_lo,ci_hi,n,r2_adj
0,Crédito popular productivo rural,const,104.2212,4.9028,0.0000,94.6120,113.8305,118,0.7987
1,Crédito popular productivo rural,t,-0.1149,0.0199,0.0000,-0.1538,-0.0759,118,0.7987
2,Crédito popular productivo rural,dtf_90d,-4.5798,0.4487,0.0000,-5.4592,-3.7004,118,0.7987
3,Crédito popular productivo rural,ibr_overnight,-0.7745,0.6362,0.2235,-2.0214,0.4725,118,0.7987
4,Crédito popular productivo rural,tes_1y,0.3583,0.4028,0.3737,-0.4311,1.1478,118,0.7987
5,Crédito popular productivo rural,tasa_finagro,0.0950,0.0934,0.3088,-0.0880,0.2781,118,0.7987
6,Crédito popular productivo urbano,const,96.8863,3.1805,0.0000,90.6526,103.1199,118,0.9435
7,Crédito popular productivo urbano,t,-0.0290,0.0119,0.0146,-0.0523,-0.0057,118,0.9435
8,Crédito popular productivo urbano,dtf_90d,-4.3237,0.3117,0.0000,-4.9347,-3.7128,118,0.9435
9,Crédito popular productivo urbano,ibr_overnight,0.6789,0.3924,0.0835,-0.0900,1.4479,118,0.9435


In [67]:
pivot_betas = (betas[betas["var"].isin(NAMES)]
                 .pivot_table(index="producto", columns="var", values="beta")
                 .round(4))
pivot_betas["r2_adj"] = betas.groupby("producto")["r2_adj"].first().round(3)
pivot_betas["n"] = betas.groupby("producto")["n"].first()
pivot_betas[NAMES + ["r2_adj", "n"]]

var,t,dtf_90d,ibr_overnight,tes_1y,tasa_finagro,r2_adj,n
producto,,,,,,,
Crédito popular productivo rural,-0.1149,-4.5798,-0.7745,0.3583,0.0950,0.799,118
Crédito popular productivo urbano,-0.0290,-4.3237,0.6789,-0.1958,-0.0163,0.943,118
Crédito productivo de mayor monto,-0.0023,-0.0647,-0.0967,-0.0101,0.2438,0.205,118
Crédito productivo rural,0.0206,0.3936,0.8005,0.0719,0.6828,0.747,118
Crédito productivo urbano,0.0694,-0.8001,1.8443,-0.1912,0.2593,0.530,118


In [68]:
def forest_plot(df_var, beta_col="beta", lo_col="ci_lo", hi_col="ci_hi",
                title="", x_title="β"):
    points = (alt.Chart(df_var)
              .mark_point(filled=True, size=120, color="black")
              .encode(
                  x=alt.X(f"{beta_col}:Q", title=x_title),
                  y=alt.Y("producto:N", sort=alt.SortField(beta_col), title=None),
                  tooltip=["producto:N",
                           alt.Tooltip(f"{beta_col}:Q", format=".3f"),
                           alt.Tooltip(f"{lo_col}:Q",   format=".3f"),
                           alt.Tooltip(f"{hi_col}:Q",   format=".3f"),
                           alt.Tooltip("p:Q", format=".4f")]))
    err = (alt.Chart(df_var).mark_rule()
           .encode(x=f"{lo_col}:Q", x2=f"{hi_col}:Q",
                   y=alt.Y("producto:N", sort=alt.SortField(beta_col))))
    zero = (alt.Chart(pd.DataFrame({"x": [0]}))
            .mark_rule(color="gray", strokeDash=[3, 3]).encode(x="x:Q"))
    return (points + err + zero).properties(width=600, height=200, title=title)

# β_time anualizado
bt = betas[betas["var"] == "t"].copy()
bt["beta_anual"]  = bt["beta"]  * 52
bt["lo_anual"]    = bt["ci_lo"] * 52
bt["hi_anual"]    = bt["ci_hi"] * 52

forest_plot(bt, beta_col="beta_anual", lo_col="lo_anual", hi_col="hi_anual",
            title="β_time anualizado por producto (controlando por DTF/IBR/TES + tasa_finagro) — CI 95%",
            x_title="β_time anualizado (pp/año)")

alt.LayerChart(...)

In [69]:
# β_finagro: pass-through del ancla Finagro al costo total del crédito
bf = betas[betas["var"] == "tasa_finagro"].copy()

forest_plot(bf, beta_col="beta", lo_col="ci_lo", hi_col="ci_hi",
            title="β_finagro por producto (proxy de pass-through, lectura comparativa) — CI 95%",
            x_title="β_finagro (pp de tasa_total por pp de tasa_finagro)")

alt.LayerChart(...)

In [70]:
X_macro = panel[["dtf_90d", "ibr_overnight", "tes_1y"]].dropna().values
vifs = pd.DataFrame({
    "var":  ["dtf_90d", "ibr_overnight", "tes_1y"],
    "VIF":  [variance_inflation_factor(X_macro, i) for i in range(X_macro.shape[1])],
})
print("VIF de las macro (>10 = colinealidad alta):")
vifs

VIF de las macro (>10 = colinealidad alta):


,var,VIF
0,dtf_90d,661.829431
1,ibr_overnight,552.097802
2,tes_1y,87.321531


### Robustez: modelo reducido sin IBR/TES

Si IBR y DTF están muy correlacionadas, los betas individuales del modelo completo son ruidosos. Reestimamos con solo `t + dtf_90d + tasa_finagro` y comparamos β_time y β_finagro contra el modelo completo. Si son cualitativamente iguales (signo, orden entre productos, magnitud razonable) el hallazgo es robusto.

In [71]:
NAMES_R = ["t", "dtf_90d", "tasa_finagro"]

rows_r = []
for prod in PRODUCTOS:
    sub = panel[panel["producto_de_credito_red"] == prod].sort_values("fecha_corte")
    if len(sub) < len(NAMES_R) + 5:
        continue
    res = fit_one(sub, formula_cols=tuple(NAMES_R))
    tbl = coef_table(res, NAMES_R).assign(producto=prod)
    rows_r.append(tbl)
betas_r = pd.concat(rows_r, ignore_index=True)

comp = (
    betas[betas["var"].isin(["t", "tasa_finagro"])]
        [["producto", "var", "beta", "p"]]
        .rename(columns={"beta": "beta_full", "p": "p_full"})
        .merge(
            betas_r[betas_r["var"].isin(["t", "tasa_finagro"])]
                [["producto", "var", "beta", "p"]]
                .rename(columns={"beta": "beta_red", "p": "p_red"}),
            on=["producto", "var"], how="outer"
        )
)
comp.round(4).sort_values(["var", "producto"])

,producto,var,beta_full,p_full,beta_red,p_red
0,Crédito popular productivo rural,t,-0.1149,0.0000,-0.0914,0.0000
2,Crédito popular productivo urbano,t,-0.0290,0.0146,-0.0466,0.0000
4,Crédito productivo de mayor monto,t,-0.0023,0.8664,-0.0007,0.9071
6,Crédito productivo rural,t,0.0206,0.1007,0.0075,0.3335
8,Crédito productivo urbano,t,0.0694,0.0000,0.0300,0.0082
1,Crédito popular productivo rural,tasa_finagro,0.0950,0.3088,0.1067,0.2193
3,Crédito popular productivo urbano,tasa_finagro,-0.0163,0.7115,-0.0256,0.5874
5,Crédito productivo de mayor monto,tasa_finagro,0.2438,0.0000,0.2450,0.0000
7,Crédito productivo rural,tasa_finagro,0.6828,0.0000,0.7097,0.0000
9,Crédito productivo urbano,tasa_finagro,0.2593,0.0000,0.2652,0.0004


### Mann-Kendall: tendencia monótona (no paramétrico)

Sobre la **serie cruda** (¿hay tendencia monótona en `tasa_total`?) y sobre el **residuo** del modelo completo (¿sobrevive la tendencia después de absorber macro y `tasa_finagro`?). Si rural muestra `decreasing` significativo en ambas → muy fuerte la evidencia.

In [72]:
mk_rows = []
for prod in PRODUCTOS:
    sub = panel[panel["producto_de_credito_red"] == prod].sort_values("fecha_corte")
    if len(sub) < len(NAMES) + 5:
        continue
    raw = mk.original_test(sub["tasa_total"].values)

    res = fit_one(sub)
    Xs = sm.add_constant(sub[NAMES].values)
    resid = sub["tasa_total"].values - res.predict(Xs)
    res_mk = mk.original_test(resid)

    mk_rows.append({
        "producto":     prod,
        "raw_trend":    raw.trend,    "raw_p":   round(raw.p, 4),    "raw_tau":   round(raw.Tau, 3),
        "resid_trend":  res_mk.trend, "resid_p": round(res_mk.p, 4), "resid_tau": round(res_mk.Tau, 3),
    })
pd.DataFrame(mk_rows)

,producto,raw_trend,raw_p,raw_tau,resid_trend,resid_p,resid_tau
0,Crédito popular productivo rural,increasing,0.0000,0.347,no trend,0.8927,-0.009
1,Crédito popular productivo urbano,increasing,0.0000,0.671,no trend,0.8890,0.009
2,Crédito productivo de mayor monto,no trend,0.4077,0.052,no trend,0.6418,-0.029
3,Crédito productivo rural,decreasing,0.0000,-0.302,no trend,0.4882,0.043
4,Crédito productivo urbano,increasing,0.0000,0.416,no trend,0.8927,0.009


### Lectura

_(Llenar tras correr las celdas de arriba)_

- **β_time anualizado**: producto con la pendiente más negativa significativa: crédito popular productivo rural
- **β_finagro**: orden entre productos: crédito productivo rural, crédito productivo urbano, crédito productivo de mayor monto. Cero para productivo popular urbano y productivo popular rural.
- **Robustez (modelo reducido)**: signos y orden entre productos coinciden: SÍ
- **Mann-Kendall**: rural `raw_trend` = decreasing, `resid_trend` = no trend, raw p = 0.000, resid_p = 0.8927
- **Conclusión**: ...

## 10. ¿Cuánto explica el macro la tasa privada (sin Finagro)?

Para aislar la dinámica del crédito privado, sacamos el componente Finagro de la variable dependiente: $Y = \text{tasa\_no\_finagro}$ (de `wide`). Estimamos por producto solo con macro (sin tendencia lineal, para no contaminar el R²):

$$\text{tasa\_no\_finagro}_{t,p} = \alpha_p + \beta_{DTF,p}\,\text{DTF}_t + \beta_{IBR,p}\,\text{IBR}_t + \beta_{TES,p}\,\text{TES}_t + \varepsilon_{t,p}$$

**Métrica clave: R²**, no los betas individuales. La colinealidad macro (VIF altísimo) rompe los betas DTF/IBR/TES individuales pero **no afecta la capacidad explicativa conjunta**.

Lectura:
- **R² alto** → las tasas privadas siguen el ciclo Banrep; no hay dinámica propia.
- **R² bajo** → algo no-macro mueve las tasas privadas (composición Finagro vía IBC, competencia, mix interno).

Después del modelo macro corremos una regresión separada del **residuo contra `t`** para ver si queda una tendencia lineal sin explicar — esto evita que `t` se lleve varianza compartida con macro y deja la R² principal limpia.

In [ ]:
NAMES_M = ["dtf_90d", "ibr_overnight", "tes_1y"]

panel_priv = wide.merge(
    macro[["fecha_corte", "dtf_90d", "ibr_overnight", "tes_1y"]],
    on="fecha_corte", how="left",
)
panel_priv["t"] = ((panel_priv["fecha_corte"] - panel_priv["fecha_corte"].min()).dt.days // 7).astype(int)
panel_priv = panel_priv.dropna(subset=NAMES_M + ["tasa_no_finagro"])
print("panel_priv shape:", panel_priv.shape)

rows_m = []
for prod in PRODUCTOS:
    sub = panel_priv[panel_priv["producto_de_credito_red"] == prod].sort_values("fecha_corte")
    if len(sub) < len(NAMES_M) + 5:
        continue
    X = sm.add_constant(sub[NAMES_M].values)
    y = sub["tasa_no_finagro"].values
    res = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 4})
    rows_m.append({
        "producto":   prod,
        "n":          int(res.nobs),
        "r2":         res.rsquared,
        "r2_adj":     res.rsquared_adj,
        "beta_dtf":   res.params[1],
        "beta_ibr":   res.params[2],
        "beta_tes":   res.params[3],
    })

res_macro = pd.DataFrame(rows_m).round(4).sort_values("r2", ascending=False)
res_macro

In [ ]:
r2_chart = (
    alt.Chart(res_macro)
    .mark_bar()
    .encode(
        x=alt.X("r2:Q", title="R² del modelo macro", scale=alt.Scale(domain=[0, 1])),
        y=alt.Y("producto:N", sort=alt.SortField("r2", order="descending"), title=None),
        color=alt.Color("r2:Q", scale=alt.Scale(scheme="redyellowgreen"), legend=None),
        tooltip=["producto:N",
                 alt.Tooltip("r2:Q",     format=".3f"),
                 alt.Tooltip("r2_adj:Q", format=".3f"),
                 alt.Tooltip("n:Q")],
    )
    .properties(width=600, height=200,
                title="¿Cuánto del movimiento de tasa_no_finagro explica el macro?")
)
r2_chart

In [ ]:
# Tendencia residual: regresión del residuo (tasa_no_finagro - macro) contra t, por producto.
# Mide cuánto del residuo es tendencia lineal vs ruido.
rows_resid = []
for prod in PRODUCTOS:
    sub = panel_priv[panel_priv["producto_de_credito_red"] == prod].sort_values("fecha_corte")
    if len(sub) < len(NAMES_M) + 5:
        continue
    X = sm.add_constant(sub[NAMES_M].values)
    res_macro_fit = sm.OLS(sub["tasa_no_finagro"].values, X).fit()
    resid = sub["tasa_no_finagro"].values - res_macro_fit.predict(X)

    Xt = sm.add_constant(sub["t"].values)
    res_t = sm.OLS(resid, Xt).fit(cov_type="HAC", cov_kwds={"maxlags": 4})
    bt   = res_t.params[1]
    se_t = res_t.bse[1]
    ci   = res_t.conf_int()[1]

    rows_resid.append({
        "producto":         prod,
        "beta_t_anual":     bt * 52,
        "se_t_anual":       se_t * 52,
        "p":                res_t.pvalues[1],
        "ci_lo":            ci[0] * 52,
        "ci_hi":            ci[1] * 52,
        "r2_resid_t":       res_t.rsquared,
    })

res_residtrend = pd.DataFrame(rows_resid).round(4).sort_values("beta_t_anual")
print("Tendencia lineal residual (anualizada, pp/año) — sobre el residuo del modelo macro:")
print(res_residtrend.to_string(index=False))

# Forest plot del β_t residual
fr = res_residtrend.rename(columns={"beta_t_anual": "beta"})
forest_plot(fr,
            title="Tendencia residual: β_t anualizado (residuo del modelo macro vs t) — CI 95%",
            x_title="β_t anualizado (pp/año)")

### Lectura

_(Llenar tras correr)_

**Capacidad explicativa del macro (R²):**
- Producto con R² más alto: ... (macro explica casi todo)
- Producto con R² más bajo: ... (algo no-macro mueve las tasas)

**Tendencia residual (β_t anualizado, sobre el residuo):**
- Producto con pendiente residual más negativa significativa: ...
- ¿Coincide con el producto de mayor efecto Finagro sobre el IBC (§11)? SÍ / NO

**Conclusión**: ...

### Sanity check: real vs predicho por macro, tasa total e IBR

Para cada producto:
- **Tasa no-Finagro (real)** — la serie privada observada.
- **Tasa no-Finagro predicha por macro** — ajuste OLS de `tasa_no_finagro ~ DTF + IBR + TES` (sin t), por producto. La diferencia entre real y predicha es la parte **no explicada por macro** que el β_time absorbe.
- **Tasa total (real)** — promedio ponderado del producto, incluyendo Finagro.
- **IBR overnight** — referencia macro.

Eje Y independiente por panel (los rangos varían mucho entre productos).

In [ ]:
MACRO_X = ["dtf_90d", "ibr_overnight", "tes_1y"]

# Unir tasa_total (de panel §9) con tasa_no_finagro y macro (de panel_priv §10)
viz = (panel_priv[["fecha_corte", "producto_de_credito_red",
                   "tasa_no_finagro", "tasa_finagro",
                   "dtf_90d", "ibr_overnight", "tes_1y"]]
       .merge(panel[["fecha_corte", "producto_de_credito_red", "tasa_total"]],
              on=["fecha_corte", "producto_de_credito_red"], how="inner"))

# Predicción por producto con OLS macro-only (sin t)
viz["tasa_no_finagro_pred"] = np.nan
for prod in PRODUCTOS:
    m = viz["producto_de_credito_red"] == prod
    sub = viz.loc[m].sort_values("fecha_corte")
    X = sm.add_constant(sub[MACRO_X].values)
    res = sm.OLS(sub["tasa_no_finagro"].values, X).fit()
    viz.loc[m, "tasa_no_finagro_pred"] = res.predict(X)

series_map = {
    "tasa_no_finagro":      "Tasa no-Finagro (real)",
    "tasa_total":           "Tasa total (real)",
    "tasa_no_finagro_pred": "Tasa no-Finagro predicha por macro",
    "ibr_overnight":        "IBR overnight",
}

long_viz = (viz.melt(id_vars=["fecha_corte", "producto_de_credito_red"],
                     value_vars=list(series_map),
                     var_name="serie", value_name="valor"))
long_viz["serie"] = long_viz["serie"].map(series_map)
long_viz["tipo"]  = long_viz["serie"].map({
    "Tasa no-Finagro (real)":              "Real",
    "Tasa total (real)":                   "Real",
    "Tasa no-Finagro predicha por macro":  "Predicha",
    "IBR overnight":                       "Macro",
})

color_scale = alt.Scale(
    domain=["Tasa no-Finagro (real)", "Tasa total (real)",
            "Tasa no-Finagro predicha por macro", "IBR overnight"],
    range=["#d62728", "#1f77b4", "#ff7f0e", "#2ca02c"],
)
dash_scale = alt.Scale(
    domain=["Real", "Predicha", "Macro"],
    range=[[1, 0], [5, 4], [2, 2]],
)

chart = (
    alt.Chart(long_viz)
    .mark_line()
    .encode(
        x=alt.X("fecha_corte:T", title="Fecha"),
        y=alt.Y("valor:Q", title="Tasa (%)", axis=alt.Axis(format=".1f")),
        color=alt.Color("serie:N", scale=color_scale, title=None),
        strokeDash=alt.StrokeDash("tipo:N", scale=dash_scale, legend=None),
        tooltip=["fecha_corte:T", "producto_de_credito_red:N", "serie:N",
                 alt.Tooltip("valor:Q", format=".2f")],
    )
    .properties(width=420, height=240)
    .facet(facet=alt.Facet("producto_de_credito_red:N", title=None), columns=2)
    .resolve_scale(y="independent")
)
chart

## 11. Efecto Finagro sobre el IBC y el techo de usura

Bajo la metodología del IBC, `tasa_total` es el promedio ponderado por monto que la SFC publica y que fija el techo de usura del mes siguiente. Como Finagro coloca volumen a tasas bajas (~13%), **deprime el IBC mecánicamente** por su peso en la ponderación.

Cantidad clave:

$$\text{efecto}_F = \text{tasa\_total} - \text{tasa\_no\_finagro} = -w_F \cdot \text{gap}$$

Es cuántos pp **más bajo** queda el IBC del producto por la presencia de Finagro. El techo de usura sin Finagro sería ~$1.5 \times \text{tasa\_no\_finagro}$ vs el observado $\approx 1.5 \times \text{tasa\_total}$.

⚠ El factor `1.5×` aplica a Crédito de Consumo y Ordinario; para microcrédito y otros productos las reglas SFC pueden diferir. La metodología exacta vive en [`Análisis/metodología de cálculo de la tasa de interés.docx`](metodología%20de%20cálculo%20de%20la%20tasa%20de%20interés.docx).

In [ ]:
FACTOR_USURA = 1.5  # techo = FACTOR_USURA × IBC

viz_ibc = viz.copy()
viz_ibc["efecto_F"]            = viz_ibc["tasa_total"] - viz_ibc["tasa_no_finagro"]
viz_ibc["techo_con_finagro"]   = FACTOR_USURA * viz_ibc["tasa_total"]
viz_ibc["techo_sin_finagro"]   = FACTOR_USURA * viz_ibc["tasa_no_finagro"]
viz_ibc["delta_techo"]         = viz_ibc["techo_con_finagro"] - viz_ibc["techo_sin_finagro"]

def bootstrap_ci(x, n_boot=2000, alpha=0.05, seed=42):
    rng = np.random.default_rng(seed)
    x = np.asarray(x)
    means = rng.choice(x, size=(n_boot, len(x)), replace=True).mean(axis=1)
    return np.quantile(means, [alpha / 2, 1 - alpha / 2])

rows = []
for prod in PRODUCTOS:
    sub = viz_ibc[viz_ibc["producto_de_credito_red"] == prod]
    eff = sub["efecto_F"].values
    lo, hi = bootstrap_ci(eff)
    rows.append({
        "producto":              prod,
        "efecto_F_pp_promedio":  round(eff.mean(), 2),
        "efecto_F_ci95_lo":      round(lo, 2),
        "efecto_F_ci95_hi":      round(hi, 2),
        "delta_techo_pp":        round(sub["delta_techo"].mean(), 2),
        "n":                     len(sub),
    })
resumen_ibc = pd.DataFrame(rows).sort_values("efecto_F_pp_promedio")
resumen_ibc

In [ ]:
# Serie de tiempo del efecto Finagro sobre el IBC, por producto
viz_ibc["cero"] = 0.0  # columna constante para la regla de cero (misma data => permite facet+layer)

base = alt.Chart(viz_ibc).encode(
    x=alt.X("fecha_corte:T", title="Fecha"),
)

line_eff = base.mark_line(color="#1f77b4").encode(
    y=alt.Y("efecto_F:Q", title="Efecto Finagro sobre IBC (pp)",
            axis=alt.Axis(format=".1f")),
    tooltip=["fecha_corte:T", "producto_de_credito_red:N",
             alt.Tooltip("efecto_F:Q", format=".2f"),
             alt.Tooltip("tasa_total:Q", format=".2f"),
             alt.Tooltip("tasa_no_finagro:Q", format=".2f")],
)

zero = base.mark_rule(color="black", strokeDash=[3, 3]).encode(
    y="cero:Q"
)

(line_eff + zero).properties(width=420, height=220).facet(
    facet=alt.Facet("producto_de_credito_red:N", title=None), columns=2
).resolve_scale(y="independent")

In [ ]:
# Contrafactual: techo de usura con vs sin Finagro
techo_long = viz_ibc.melt(
    id_vars=["fecha_corte", "producto_de_credito_red"],
    value_vars=["techo_con_finagro", "techo_sin_finagro"],
    var_name="escenario", value_name="techo",
)
techo_long["escenario"] = techo_long["escenario"].replace({
    "techo_con_finagro": "Observado (1.5 × IBC)",
    "techo_sin_finagro": "Contrafactual sin Finagro (1.5 × tasa_no_finagro)",
})

techo_chart = (
    alt.Chart(techo_long)
    .mark_line()
    .encode(
        x=alt.X("fecha_corte:T", title="Fecha"),
        y=alt.Y("techo:Q", title="Techo de usura (% E.A.)",
                axis=alt.Axis(format=".0f")),
        color=alt.Color("escenario:N",
            scale=alt.Scale(
                domain=["Observado (1.5 × IBC)",
                        "Contrafactual sin Finagro (1.5 × tasa_no_finagro)"],
                range=["#1f77b4", "#d62728"],
            ),
            title=None),
        tooltip=["fecha_corte:T", "producto_de_credito_red:N", "escenario:N",
                 alt.Tooltip("techo:Q", format=".2f")],
    )
    .properties(width=420, height=240)
    .facet(facet=alt.Facet("producto_de_credito_red:N", title=None), columns=2)
    .resolve_scale(y="independent")
)
techo_chart

## 12. Pass-through directo: ¿la tasa Finagro mueve la tasa privada?

La sección 9 estimaba `tasa_total ~ tasa_finagro`, pero el caveat reconoce que el coeficiente mezcla **composición** (canal mecánico de la sec. 11: como `tasa_total ≈ w·tasa_finagro + (1−w)·tasa_no_finagro`, la tasa_finagro entra por construcción) con **pass-through** real al mercado privado (cambios en `tasa_no_finagro`). No se identifica limpio el canal de comportamiento.

Aquí usamos directamente `tasa_no_finagro` como dependiente — así `tasa_finagro` solo puede afectarla vía un mecanismo económico, no por aritmética del promedio.

**Modelo:**

$$\text{tasa\_no\_finagro}_{t,p} = \alpha_p + \gamma_0^p \cdot \text{tasa\_finagro}_{t,p} + \gamma_4^p \cdot \text{tasa\_finagro}_{t-4,p} + \beta_t^p\,t + \beta_{DTF,p}\,\text{DTF}_t + \beta_{IBR,p}\,\text{IBR}_t + \beta_{TES,p}\,\text{TES}_t + \varepsilon_{t,p}$$

Con SE Newey-West (HAC, maxlags=8). Lags en semanas — el panel es semanal con cierre de viernes.

**Hipótesis sobre lags** (responden a *por qué* esperaríamos transmisión):

- **γ₀ — canal de competencia directa**. Los bancos privados ven precio Finagro y reaccionan. Lag corto, casi contemporáneo. Si Finagro compite *codo a codo* con la banca por el mismo cliente, γ₀ > 0 y significativo.
- **γ₄ (~1 mes) — canal del techo de usura**. La cadena: tasa Finagro de la semana s → entra al IBC publicado por SFC al inicio del mes siguiente → fija el techo de usura del mes subsiguiente → la tasa privada se ajusta al nuevo techo. Si este canal domina, γ₄ > 0 y γ₀ ≈ 0.
- Si ambos lags son significativos, ambos canales operan. Si ninguno lo es, Finagro deprime el IBC solo mecánicamente (sec. 11) sin disciplinar al mercado privado.

**Lectura cross-product (clave para identificación):** si γ crece con el peso de Finagro (w_F) entre los 5 productos, eso es evidencia consistente con un canal causal de competencia. Si γ es plano respecto a w_F, la asociación es ruido.

In [ ]:
# Construir panel para pass-through: dependiente = tasa_no_finagro, regresor = tasa_finagro (contemporáneo + lag 4)
panel_pt = (panel
    .merge(wide[["fecha_corte", "producto_de_credito_red", "tasa_no_finagro"]],
           on=["fecha_corte", "producto_de_credito_red"], how="left")
    .sort_values(["producto_de_credito_red", "fecha_corte"])
    .reset_index(drop=True)
)

# Lags por producto (no contaminar entre productos). 4 semanas ≈ 1 mes, 8 semanas ≈ 2 meses.
for L in [4, 8]:
    panel_pt[f"tasa_finagro_lag{L}"] = (
        panel_pt.groupby("producto_de_credito_red")["tasa_finagro"].shift(L)
    )

NAMES_PT = ["t", "dtf_90d", "ibr_overnight", "tes_1y", "tasa_finagro", "tasa_finagro_lag4"]

panel_pt_clean = panel_pt.dropna(subset=NAMES_PT + ["tasa_no_finagro"])
print("panel_pt shape (después de dropna):", panel_pt_clean.shape)
print(panel_pt_clean.groupby("producto_de_credito_red").size().rename("n_semanas"))

# Estimar por producto con HAC, maxlags=8 (más conservador con lags incluidos)
rows_pt = []
for prod in PRODUCTOS:
    sub = panel_pt_clean[panel_pt_clean["producto_de_credito_red"] == prod].sort_values("fecha_corte")
    if len(sub) < len(NAMES_PT) + 5:
        print(f"Saltando {prod}: solo {len(sub)} obs.")
        continue
    X = sm.add_constant(sub[NAMES_PT].values)
    y = sub["tasa_no_finagro"].values
    res = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 8})
    tbl = coef_table(res, NAMES_PT).assign(
        producto=prod, n=int(res.nobs),
        r2=res.rsquared, r2_adj=res.rsquared_adj,
    )
    rows_pt.append(tbl)

betas_pt = pd.concat(rows_pt, ignore_index=True)

# Vista compacta: γ₀, γ₄ y su suma por producto
gamma = (betas_pt[betas_pt["var"].isin(["tasa_finagro", "tasa_finagro_lag4"])]
           .pivot_table(index="producto", columns="var",
                        values=["beta", "p", "ci_lo", "ci_hi"]))
gamma.columns = [f"{a}_{b}" for a, b in gamma.columns]
gamma = gamma.round(4)
gamma["gamma_total"] = (gamma["beta_tasa_finagro"] + gamma["beta_tasa_finagro_lag4"]).round(4)
print("\nCoeficientes de pass-through (γ₀, γ₄, suma) por producto:")
gamma[["beta_tasa_finagro", "p_tasa_finagro",
       "beta_tasa_finagro_lag4", "p_tasa_finagro_lag4",
       "gamma_total"]]

In [ ]:
# Robustez: ¿qué pasa con lag 8 (≈2 meses)? Estimamos especificación alternativa.
NAMES_PT_R = ["t", "dtf_90d", "ibr_overnight", "tes_1y",
              "tasa_finagro", "tasa_finagro_lag4", "tasa_finagro_lag8"]
panel_pt_R = panel_pt.dropna(subset=NAMES_PT_R + ["tasa_no_finagro"])

rows_R = []
for prod in PRODUCTOS:
    sub = panel_pt_R[panel_pt_R["producto_de_credito_red"] == prod].sort_values("fecha_corte")
    if len(sub) < len(NAMES_PT_R) + 5:
        continue
    X = sm.add_constant(sub[NAMES_PT_R].values)
    res = sm.OLS(sub["tasa_no_finagro"].values, X).fit(
        cov_type="HAC", cov_kwds={"maxlags": 8})
    tbl = coef_table(res, NAMES_PT_R).assign(producto=prod)
    rows_R.append(tbl)

betas_pt_R = pd.concat(rows_R, ignore_index=True)
gamma_R = (betas_pt_R[betas_pt_R["var"].str.startswith("tasa_finagro")]
             .pivot_table(index="producto", columns="var", values="beta").round(4))
gamma_R["suma_lags"] = gamma_R.sum(axis=1).round(4)
print("Lags 0+4+8 — coeficientes (sirve para ver si el efecto persiste a 2 meses):")
gamma_R

In [ ]:
# Forest plots: γ₀ (competencia directa) y γ₄ (vía techo de usura) por producto
g0 = betas_pt[betas_pt["var"] == "tasa_finagro"].copy()
g4 = betas_pt[betas_pt["var"] == "tasa_finagro_lag4"].copy()

fp_g0 = forest_plot(g0, title="γ₀ — pass-through contemporáneo (canal: competencia directa) — CI 95%",
                    x_title="γ₀ (pp de tasa_no_finagro por pp de tasa_finagro_t)")
fp_g4 = forest_plot(g4, title="γ₄ — pass-through con lag de 4 semanas (canal: vía techo de usura) — CI 95%",
                    x_title="γ₄ (pp de tasa_no_finagro por pp de tasa_finagro_{t−4})")

(fp_g0 & fp_g4)

In [ ]:
# Cross-product: ¿el pass-through crece con el peso de Finagro?
# w_F por (semana, producto): despejado de tasa_total = w·tasa_finagro + (1−w)·tasa_no_finagro
wp = (panel.merge(wide[["fecha_corte", "producto_de_credito_red", "tasa_no_finagro"]],
                  on=["fecha_corte", "producto_de_credito_red"], how="left")
            .assign(w_F = lambda d: (d["tasa_no_finagro"] - d["tasa_total"]) /
                                     (d["tasa_no_finagro"] - d["tasa_finagro"]))
            .replace([np.inf, -np.inf], np.nan)
            .dropna(subset=["w_F"]))
w_F_mean = wp.groupby("producto_de_credito_red")["w_F"].mean().rename("w_F").reset_index()

# Unir con γ_total (γ₀ + γ₄)
cross = (gamma[["gamma_total"]].reset_index()
           .merge(w_F_mean, left_on="producto", right_on="producto_de_credito_red")
           .drop(columns=["producto_de_credito_red"]))
print("γ_total vs w_F (peso medio de Finagro en monto desembolsado):")
print(cross.round(3).to_string(index=False))

# Scatter + tendencia
scatter = (alt.Chart(cross)
    .mark_point(filled=True, size=200, color="#1f77b4")
    .encode(
        x=alt.X("w_F:Q", title="w_F (peso Finagro promedio)", scale=alt.Scale(zero=False)),
        y=alt.Y("gamma_total:Q", title="γ_total = γ₀ + γ₄ (pass-through)"),
        tooltip=["producto:N",
                 alt.Tooltip("w_F:Q", format=".2f"),
                 alt.Tooltip("gamma_total:Q", format=".3f")],
    ))
labels = (alt.Chart(cross).mark_text(align="left", dx=8, dy=-8, fontSize=11)
            .encode(x="w_F:Q", y="gamma_total:Q", text="producto:N"))
fit = scatter.transform_regression("w_F", "gamma_total").mark_line(strokeDash=[4,4], color="gray")
zero = alt.Chart(pd.DataFrame({"y":[0]})).mark_rule(color="black", strokeDash=[2,2]).encode(y="y:Q")

(scatter + labels + fit + zero).properties(
    width=600, height=400,
    title="Cross-product: pass-through crece con el peso de Finagro? (evidencia identificación)"
)

### Lectura

_(Llenar tras correr las celdas de arriba)_

**γ₀ (canal competencia directa):**
- Producto con γ₀ más positivo y significativo: ...
- ¿Cero / no significativo en otros productos? ...

**γ₄ (canal vía techo de usura, lag 1 mes):**
- Producto con γ₄ más positivo y significativo: ...
- ¿Sobrevive cuando se agrega el lag 8? ...

**γ_total = γ₀ + γ₄:**
- Ranking entre productos: ...

**Cross-product (γ_total vs w_F):**
- Pendiente positiva → más Finagro, más pass-through → consistente con canal causal
- Pendiente plana o negativa → pass-through no escala con el peso, asociación es ruido
- Producto que más sale del patrón: ...

**Conclusión sobre la pregunta principal:**

> _¿La tasa de Finagro (limitada por políticas sociales/subsidios) impacta el IBC del crédito productivo rural?_

- **Canal mecánico (sec. 11):** efecto Finagro sobre IBC rural ≈ ... pp; sobre techo de usura ≈ ... pp. (Aritmética, cerrada.)
- **Canal de comportamiento (esta sección):** γ_total rural = ..., p-valor = ...; ¿significativo? SÍ / NO
- **Evidencia cross-product:** la pendiente de γ vs w_F es ... → la asociación ¿escala con el peso?

**Veredicto:** ...